# 02 · Thiết kế content gate và hàng chờ phân loại

Mục tiêu của notebook này là biến kết quả EDA thành một thứ tự quyết định có thể kiểm tra. Phạm vi đã chốt: loại YouTube Shorts, video giải trí và podcast; giữ video có nội dung chính là âm nhạc. Video nói chuyện/game có BGM không thuộc thư viện tải.

Notebook chỉ mô phỏng rule và audit sample; chưa tải metadata/audio và chưa tuyên bố accuracy. `manual_label` của người dùng vẫn là nguồn quyết định cao nhất ở cấp video.


In [ ]:
from pathlib import Path
from collections import Counter
import json
import re
import platform
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.ai-devkit.json').exists()), None)
if ROOT is None:
    raise RuntimeError('Mở notebook từ repo Auralytica hoặc notebooks/.')
EDA_ROOT = ROOT / 'artifacts' / 'notebook-runs' / '01_takeout_eda'
feature_files = sorted(EDA_ROOT.glob('*/signals-v2/video_features.csv'), key=lambda p: p.stat().st_mtime)
if not feature_files:
    raise FileNotFoundError('Chạy notebook 01 signals-v2 trước.')
FEATURE_FILE = feature_files[-1]
OUTPUT_DIR = FEATURE_FILE.parent / 'classification-design-v1'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
videos = pd.read_csv(FEATURE_FILE, keep_default_na=False)
print({'python': platform.python_version(), 'videos': len(videos), 'source': str(FEATURE_FILE.relative_to(ROOT))})


## 1. Mô hình tham khảo: catalog matching sau content gate

Spotify tích hợp TuneMyMusic để nhập playlist từ dịch vụ khác. Tài liệu của [TuneMyMusic](https://www.tunemymusic.com/en) mô tả matching theo ISRC/title/artist; [Soundiiz](https://support.soundiiz.com/hc/en-us/articles/360010097413-How-to-Use-Soundiiz-to-Transfer-Playlists-and-Favorites) dùng title, artist, album, duration và ISRC. Spotify Search cũng hỗ trợ filter `track`, `artist`, `album`, `isrc` trong [Web API](https://developer.spotify.com/documentation/web-api/reference/search).

Điều có thể học theo là chuẩn hóa metadata, xếp hạng candidate, giữ unmatched và ghi nhớ correction. Không thể lấy matching làm bộ lọc ban đầu vì playlist chuyển dịch vốn đã gồm bài nhạc, còn YouTube watch history chứa mọi loại video.

```text
Takeout → content gate → music candidates → canonical track matching → review/unmatched → download original audio
```


## 2. Nguồn loại trừ và độ tin cậy

`channel_labels.csv` chứa quyết định người dùng theo channel URL, không suy ra từ tên. `entertainment` và `podcast` là hard exclusion ở cấp kênh nhưng vẫn được giữ trong dữ liệu để audit. `#shorts` là proxy có độ phủ chưa biết: từ Takeout hiện tại không có URL `/shorts/` hay trường `isShort`. [YouTube cho phép Shorts dài tới ba phút từ 15/10/2024](https://support.google.com/youtube/answer/13053317?hl=en), nên duration đơn độc cũng không xác định chắc chắn. Implementation cần trường `confirmed_is_short` từ probe/metadata và giữ nguồn bằng chứng `proxy` để audit.


In [ ]:
CHANNEL_LABELS = ROOT / 'artifacts' / 'notebook-runs' / 'channel_labels.csv'
if not CHANNEL_LABELS.exists():
    raise FileNotFoundError(f'Tạo file nhãn kênh trước: {CHANNEL_LABELS}')
channel_labels = pd.read_csv(CHANNEL_LABELS, keep_default_na=False)
required = {'channel_key', 'label', 'source', 'notes'}
if not required <= set(channel_labels):
    raise ValueError(f'channel_labels.csv thiếu cột: {required - set(channel_labels)}')
allowed = {'music', 'entertainment', 'podcast', 'mixed', 'unknown'}
invalid = set(channel_labels.label) - allowed
if invalid or channel_labels.channel_key.duplicated().any():
    raise ValueError(f'Nhãn lỗi {invalid} hoặc channel_key bị trùng.')
videos = videos.merge(channel_labels[['channel_key','label','source']], on='channel_key', how='left', validate='many_to_one')
videos[['label','source']] = videos[['label','source']].fillna('')
display(channel_labels.groupby(['label','source']).size().rename('channels').to_frame())
display(videos.loc[videos.label.ne('')].groupby('label').agg(videos=('video_id','size'), watch_events=('watch_count','sum')))


## 3. Tín hiệu loại podcast/giải trí và nhận video nhạc

Podcast cần kết hợp từ khóa format (`podcast`, `episode`, `tập`), ngữ cảnh nói chuyện và sau này là duration/series metadata. Từ `radio`, `live`, `cover` hoặc `episode` một mình đều có phản ví dụ âm nhạc. Kênh `mixed` luôn đi review ở cấp video.

Quy tắc bên dưới tạo trạng thái điều phối, không tạo ground truth: hard exclusion đã được người dùng xác nhận; nguồn Topic/library là strong music; các ca còn lại được xếp vào review.


In [ ]:
PODCAST_FORMAT = r'(?<!\w)(?:podcast|episode|ep\.?\s*\d+|talk[ -]?show|audiobook)(?!\w)|phỏng vấn|tâm sự|chuyện trò|tọa đàm|tập\s*\d+'
SERIAL_ENTERTAINMENT = r'(?<!\w)(?:part|episode|ep)\s*\d+(?!\w)|phần\s*\d+|tập\s*\d+|review|recap|anime|manga|movie|phim'
videos['signal_podcast_format'] = videos.title_norm.str.contains(PODCAST_FORMAT, regex=True)
videos['signal_explicit_podcast'] = videos.title_norm.str.contains(r'(?<!\w)(?:podcast|audiobook)(?!\w)|tọa đàm', regex=True)
videos['signal_serial_entertainment'] = videos.title_norm.str.contains(SERIAL_ENTERTAINMENT, regex=True)
videos['suspected_short'] = videos.signal_shortform_context.astype(bool)
videos['confirmed_is_short'] = False  # Chưa có trường xác nhận trong Takeout hiện tại.
videos['excluded_channel'] = videos.label.isin(['entertainment','podcast'])
videos['podcast_review'] = videos.label.eq('podcast') | videos.signal_podcast_format
videos['hard_excluded_observed'] = videos.confirmed_is_short | videos.suspected_short | videos.excluded_channel | videos.signal_explicit_podcast
videos['strong_music_after_gate'] = videos.strong_music_evidence & ~videos.hard_excluded_observed
videos['decision'] = 'review_insufficient'
videos.loc[videos.content_music_hint | videos.signal_repeat_days, 'decision'] = 'review_music_candidate'
videos.loc[videos.signal_serial_entertainment | videos.podcast_review, 'decision'] = 'review_exclude_candidate'
videos.loc[videos.suspected_short, 'decision'] = 'review_suspected_short'
videos.loc[videos.strong_music_after_gate, 'decision'] = 'accept_strong_music'
videos.loc[videos.hard_excluded_observed, 'decision'] = 'exclude_short_channel_or_explicit_podcast'
display(videos.decision.value_counts().rename('videos').to_frame())
assert not (videos.hard_excluded_observed & videos.decision.eq('accept_strong_music')).any()
assert videos.loc[videos.excluded_channel, 'decision'].eq('exclude_short_channel_or_explicit_podcast').all()
assert videos.loc[videos.suspected_short, 'decision'].eq('exclude_short_channel_or_explicit_podcast').all()


## 4. Audit lại sample v2

Mục tiêu là tìm lý do sample cũ gây hiểu nhầm. Không đọc nhãn random holdout và không thay đổi CSV v2. Bảng audit cho biết mỗi stratum chứa bao nhiêu Shorts proxy, kênh đã xác nhận loại, podcast candidate và nguồn nhạc mạnh.


In [ ]:
DISCOVERY_FILE = FEATURE_FILE.parent / 'discovery_review.csv'
discovery = pd.read_csv(DISCOVERY_FILE, keep_default_na=False)
audit = discovery[['video_id','sampling_stratum']].merge(videos, on='video_id', validate='one_to_one')
sample_audit = audit.groupby('sampling_stratum').agg(
    sample_size=('video_id','size'), suspected_short=('suspected_short','sum'),
    confirmed_excluded_channel=('excluded_channel','sum'), podcast_review=('podcast_review','sum'),
    strong_music=('strong_music_after_gate','sum'), repeat_days=('signal_repeat_days','sum'))
display(sample_audit)
display(pd.crosstab(audit.sampling_stratum, audit.decision))
sample_audit.to_csv(OUTPUT_DIR / 'sample_v2_gate_audit.csv')


In [ ]:
channel_audit = videos.groupby(['channel_key','channel'], as_index=False).agg(
    unique_videos=('video_id','size'), watch_events=('watch_count','sum'),
    suspected_shorts=('suspected_short','sum'), podcast_candidates=('podcast_review','sum'),
    strong_music=('strong_music_after_gate','sum'), music_hints=('content_music_hint','sum'))
channel_audit['suspected_short_share'] = channel_audit.suspected_shorts / channel_audit.unique_videos
channel_audit['music_hint_share'] = channel_audit.music_hints / channel_audit.unique_videos
channel_audit = channel_audit.sort_values(['unique_videos','watch_events'], ascending=False)
display(channel_audit.head(25))
print('Bảng này dùng để đề xuất channel review; tỷ lệ cao không tự tạo nhãn entertainment/podcast.')
channel_audit.to_csv(OUTPUT_DIR / 'channel_gate_audit.csv', index=False)
videos.to_csv(OUTPUT_DIR / 'decision_preview.csv', index=False)


## 5. Kết luận thiết kế và dữ liệu còn thiếu

Thứ tự gate đề xuất:

1. Loại bản ghi không phải video/quảng cáo/unavailable.
2. Loại Shorts đã xác nhận và video có marker Shorts rõ trong Takeout; lưu lý do `proxy` để sau này đối chiếu metadata.
3. Áp dụng nhãn kênh/video do người dùng xác nhận: entertainment, podcast, music, mixed.
4. Nhận Topic/music-library sau gate.
5. Chấm tín hiệu metadata: artist/title/album/ISRC/duration, VEVO, version, nhạc cụ, description/tags/category.
6. Dùng xem lặp để tăng ưu tiên review, không tự nhận là nhạc.
7. Mọi ca mâu thuẫn/unmatched xuất hiện trong UI để sửa; correction được lưu theo video/channel.

Bước phát triển tiếp theo cần một metadata pilot trên vài trăm video discovery để đo cách xác nhận Shorts và độ hữu ích của description, duration, tags, category. Chưa cần tải audio.
